In [0]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# User inputs
# --------------------------------------------------

boundary_gap = 0.75          # distance from boundary to back of shed, m
shed_depth = 3             # internal/external depth of shed, m
roof_pitch_deg = 20          # single pitch roof angle, degrees

board_angle_deg = 35         # board overhang from vertical, degrees
board_bottom_height = 0.20   # height of bottom of board above floor, m

# Planning height limits
boundary_limit_distance = 2.0
near_boundary_max_height = 2.5
far_boundary_max_height = 3.0

# Board search resolution
lengths = np.linspace(1.0, 5.0, 801)
bottom_x_positions = np.array([boundary_gap])

# --------------------------------------------------
# Geometry functions
# --------------------------------------------------

def height_limit(x):
    """Maximum permitted height at distance x from boundary."""
    return np.where(
        x <= boundary_limit_distance,
        near_boundary_max_height,
        far_boundary_max_height
    )

def roof_height(x):
    """
    Single-pitch roof rising away from the boundary.
    Back height is chosen so the roof just respects 2.5 m at x = 2 m.
    """
    pitch = np.radians(roof_pitch_deg)

    back_height = near_boundary_max_height - np.tan(pitch) * (
        boundary_limit_distance - boundary_gap
    )

    z = back_height + np.tan(pitch) * (x - boundary_gap)

    # Shed only exists between back and front
    z = np.where((x >= boundary_gap) & (x <= boundary_gap + shed_depth), z, np.nan)

    return z

def board_coordinates(bottom_x, length):
    """
    Board leans away from the boundary.
    Bottom is closer to the boundary; top is further into the shed.
    """
    angle = np.radians(board_angle_deg)

    bottom_z = board_bottom_height
    top_x = bottom_x + length * np.sin(angle)
    top_z = bottom_z + length * np.cos(angle)

    return bottom_x, bottom_z, top_x, top_z

def fits_inside_shed(bottom_x, length):
    """Checks whether board fits inside shed and below roof."""
    bx, bz, tx, tz = board_coordinates(bottom_x, length)

    if bx < boundary_gap:
        return False
    if tx > boundary_gap + shed_depth:
        return False

    xs = np.linspace(bx, tx, 100)
    zs = np.linspace(bz, tz, 100)

    roof_zs = roof_height(xs)

    if np.any(np.isnan(roof_zs)):
        return False

    return np.all(zs <= roof_zs)

# --------------------------------------------------
# Find maximum board length
# --------------------------------------------------

best = None

for bottom_x in bottom_x_positions:
    for length in lengths:
        if fits_inside_shed(bottom_x, length):
            best = (bottom_x, length)

best_bottom_x, best_length = best
bx, bz, tx, tz = board_coordinates(best_bottom_x, best_length)

print(f"Best board length: {best_length:.2f} m")
print(f"Board angle: {board_angle_deg:.1f}° overhanging")
print(f"Bottom x-position from boundary: {best_bottom_x:.2f} m")
print(f"Top x-position from boundary: {tx:.2f} m")
print(f"Top height: {tz:.2f} m")

# --------------------------------------------------
# Plot side profile
# --------------------------------------------------
plt.figure(figsize=(12, 7))

# Planning height envelope
plt.fill_between(
    x,
    0,
    height_limit(x),
    alpha=0.15,
    label="Permitted envelope"
)

plt.plot(
    x,
    height_limit(x),
    linestyle="--",
    linewidth=2,
    label="Planning height limit"
)

# Shed floor
plt.plot(
    [boundary_gap, boundary_gap + shed_depth],
    [0, 0],
    linewidth=3,
    label="Shed floor"
)

# Back wall
plt.plot(
    [boundary_gap, boundary_gap],
    [0, roof_height(boundary_gap)],
    linewidth=3
)

# Front wall
plt.plot(
    [boundary_gap + shed_depth, boundary_gap + shed_depth],
    [0, roof_height(boundary_gap + shed_depth)],
    linewidth=3
)

# Roof
shed_x = np.linspace(boundary_gap, boundary_gap + shed_depth, 200)

plt.plot(
    shed_x,
    roof_height(shed_x),
    linewidth=4,
    label=f"Roof ({roof_pitch_deg}°)"
)

# Boundary
plt.axvline(
    0,
    linestyle=":",
    linewidth=3,
    alpha=0.8,
    label="Boundary"
)

# 2m planning line
plt.axvline(
    boundary_limit_distance,
    linestyle="-.",
    linewidth=3,
    alpha=0.8,
    label="2 m rule limit"
)

# Climbing board
plt.plot(
    [bx, tx],
    [bz, tz],
    linewidth=6,
    label=f"Board ({board_angle_deg}°)"
)

# Mark board ends
plt.plot(bx, bz, marker="o", markersize=10)
plt.plot(tx, tz, marker="s", markersize=10)

# Annotate board length
plt.annotate(
    f"{best_length:.2f} m",
    ((bx + tx)/2, (bz + tz)/2),
    xytext=(10, 10),
    textcoords="offset points"
)

plt.xlabel("Distance from boundary (m)")
plt.ylabel("Height (m)")
plt.title("Climbing Shed Side Profile")

plt.axis("equal")
plt.grid(True, linestyle=":", alpha=0.5)

plt.legend(loc="upper left")

plt.tight_layout()
plt.show()

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

# --------------------------------------------------
# User inputs
# --------------------------------------------------

boundary_gap = 0.50
target_board_length = 3.0        # climbing length of board, m
top_clearance_to_front_wall = 1.0 # space beyond top of board, m
roof_pitch_deg = 25

board_angle_deg = 30
board_bottom_height = 0.20

top_clearance_to_front_wall = 1.0   # NEW: space from top of board to front wall, m

boundary_limit_distance = 2.0
near_boundary_max_height = 2.5
far_boundary_max_height = 3.0

lengths = np.linspace(1.0, 5.0, 801)
bottom_x_positions = np.array([boundary_gap])

# --------------------------------------------------
# Geometry functions
# --------------------------------------------------

def height_limit(x):
    return np.where(
        x <= boundary_limit_distance,
        near_boundary_max_height,
        far_boundary_max_height
    )

def roof_height(x):
    pitch = np.radians(roof_pitch_deg)

    back_height = near_boundary_max_height - np.tan(pitch) * (
        boundary_limit_distance - boundary_gap
    )

    z = back_height + np.tan(pitch) * (x - boundary_gap)

    return np.where(
        (x >= boundary_gap) & (x <= boundary_gap + shed_depth),
        z,
        np.nan
    )

def board_coordinates(bottom_x, length):
    angle = np.radians(board_angle_deg)

    bottom_z = board_bottom_height
    top_x = bottom_x + length * np.sin(angle)
    top_z = bottom_z + length * np.cos(angle)

    return bottom_x, bottom_z, top_x, top_z

def fits_inside_shed(bottom_x, length):
    bx, bz, tx, tz = board_coordinates(bottom_x, length)

    front_wall_x = boundary_gap + shed_depth

    if bx < boundary_gap:
        return False

    if tx > front_wall_x - top_clearance_to_front_wall:
        return False

    xs = np.linspace(bx, tx, 100)
    zs = np.linspace(bz, tz, 100)

    roof_zs = roof_height(xs)

    if np.any(np.isnan(roof_zs)):
        return False

    return np.all(zs <= roof_zs)

def add_dimension(ax, x1, y1, x2, y2, text, text_offset=(0, 0)):
    ax.annotate(
        "",
        xy=(x1, y1),
        xytext=(x2, y2),
        arrowprops=dict(arrowstyle="<->", linewidth=1.5)
    )
    ax.text(
        (x1 + x2) / 2 + text_offset[0],
        (y1 + y2) / 2 + text_offset[1],
        text,
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8)
    )

# --------------------------------------------------
# Find maximum board length
# --------------------------------------------------

best = None

for bottom_x in bottom_x_positions:
    for length in lengths:
        if fits_inside_shed(bottom_x, length):
            best = (bottom_x, length)

if best is None:
    raise ValueError("No board fits with the current shed depth, roof pitch and clearance.")

# --------------------------------------------------
# Set board and shed geometry from target board length
# --------------------------------------------------

best_length = target_board_length

bx, bz, tx, tz = board_coordinates(boundary_gap, best_length)

front_wall_x = tx + top_clearance_to_front_wall
shed_depth = front_wall_x - boundary_gap

print(f"Board length: {best_length:.2f} m")
print(f"Board angle: {board_angle_deg:.1f}° overhanging")
print(f"Roof pitch: {roof_pitch_deg:.1f}°")
print(f"Shed depth required: {shed_depth:.2f} m")
print(f"Boundary gap: {boundary_gap:.2f} m")
print(f"Top-to-front-wall clearance: {top_clearance_to_front_wall:.2f} m")
print(f"Bottom x-position from boundary: {bx:.2f} m")
print(f"Top x-position from boundary: {tx:.2f} m")
print(f"Top height: {tz:.2f} m")

# --------------------------------------------------
# Plot side profile
# --------------------------------------------------

x = np.linspace(0, boundary_gap + shed_depth + 0.75, 500)
shed_x = np.linspace(boundary_gap, boundary_gap + shed_depth, 200)

fig, ax = plt.subplots(figsize=(13, 8))

# Planning envelope
ax.fill_between(
    x,
    0,
    height_limit(x),
    alpha=0.12,
    label="Permitted planning envelope"
)

ax.plot(
    x,
    height_limit(x),
    linestyle="--",
    linewidth=2,
    label="Planning height limit"
)

# Shed
ax.plot(
    [boundary_gap, front_wall_x],
    [0, 0],
    linewidth=3,
    label="Shed floor"
)

ax.plot(
    [boundary_gap, boundary_gap],
    [0, roof_height(boundary_gap)],
    linewidth=3,
    label="Back wall"
)

ax.plot(
    [front_wall_x, front_wall_x],
    [0, roof_height(front_wall_x)],
    linewidth=3,
    label="Front wall"
)

ax.plot(
    shed_x,
    roof_height(shed_x),
    linewidth=4,
    label=f"Single-pitch roof ({roof_pitch_deg}°)"
)

# Boundary and 2m line
ax.axvline(
    0,
    linestyle=":",
    linewidth=3,
    label="Boundary"
)

ax.axvline(
    boundary_limit_distance,
    linestyle="-.",
    linewidth=3,
    label="2 m from boundary"
)

# Climbing board
ax.plot(
    [bx, tx],
    [bz, tz],
    linewidth=6,
    label=f"Climbing board ({board_angle_deg}° overhang)"
)

ax.plot(bx, bz, marker="o", markersize=9)
ax.plot(tx, tz, marker="s", markersize=9)

# Clearance zone
ax.plot(
    [tx, front_wall_x],
    [tz, tz],
    linestyle=":",
    linewidth=2,
    label="Top-to-front-wall clearance"
)

# --------------------------------------------------
# Dimension annotations
# --------------------------------------------------

# Boundary gap
add_dimension(
    ax,
    0, -0.25,
    boundary_gap, -0.25,
    f"boundary gap\n{boundary_gap:.2f} m"
)

# Shed depth
add_dimension(
    ax,
    boundary_gap, -0.50,
    front_wall_x, -0.50,
    f"shed depth\n{shed_depth:.2f} m"
)

# 2 m planning distance
add_dimension(
    ax,
    0, -0.75,
    boundary_limit_distance, -0.75,
    "2 m planning zone"
)

# Board length
add_dimension(
    ax,
    bx, bz,
    tx, tz,
    f"board length\n{best_length:.2f} m",
    text_offset=(0.15, 0.15)
)

# Top clearance
add_dimension(
    ax,
    tx, tz + 0.15,
    front_wall_x, tz + 0.15,
    f"top clearance\n{actual_top_clearance:.2f} m"
)

# Heights
add_dimension(
    ax,
    front_wall_x + 0.15, 0,
    front_wall_x + 0.15, roof_height(front_wall_x),
    f"front height\n{roof_height(front_wall_x):.2f} m"
)

add_dimension(
    ax,
    boundary_gap - 0.15, 0,
    boundary_gap - 0.15, roof_height(boundary_gap),
    f"back height\n{roof_height(boundary_gap):.2f} m"
)

# Board angle arc
arc = Arc(
    (bx, bz),
    0.8,
    0.8,
    angle=0,
    theta1=90 - board_angle_deg,
    theta2=90,
    linewidth=1.5
)
ax.add_patch(arc)

ax.text(
    bx + 0.35,
    bz + 0.55,
    f"{board_angle_deg}°\nfrom vertical",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8)
)

# Roof pitch label
ax.text(
    boundary_gap + shed_depth * 0.55,
    roof_height(boundary_gap + shed_depth * 0.55) + 0.15,
    f"roof pitch {roof_pitch_deg}°",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8)
)

# --------------------------------------------------
# Formatting
# --------------------------------------------------

ax.set_xlabel("Distance from boundary (m)")
ax.set_ylabel("Height (m)")
ax.set_title("Climbing Shed Side Profile")

ax.set_xlim(-0.2, boundary_gap + shed_depth + 0.8)
ax.set_ylim(-0.9, max(far_boundary_max_height, roof_height(front_wall_x)) + 0.5)

ax.set_aspect("equal", adjustable="box")
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

# --------------------------------------------------
# User inputs
# --------------------------------------------------

boundary_gap = 0.50
target_board_length = 3.0
top_clearance_to_front_wall = 1.0

roof_pitch_deg = 20

board_angle_deg = 35
board_bottom_height = 0.2

boundary_limit_distance = 2.0
near_boundary_max_height = 2.5
far_boundary_max_height = 3.0

# --------------------------------------------------
# Functions
# --------------------------------------------------

def height_limit(x):
    return np.where(
        x <= boundary_limit_distance,
        near_boundary_max_height,
        far_boundary_max_height
    )

def board_coordinates(bottom_x, length):
    angle = np.radians(board_angle_deg)

    bottom_z = board_bottom_height
    top_x = bottom_x + length * np.sin(angle)
    top_z = bottom_z + length * np.cos(angle)

    return bottom_x, bottom_z, top_x, top_z

def roof_height(x, front_wall_x):
    pitch = np.radians(roof_pitch_deg)

    back_height = near_boundary_max_height - np.tan(pitch) * (
        boundary_limit_distance - boundary_gap
    )

    z = back_height + np.tan(pitch) * (x - boundary_gap)

    return np.where(
        (x >= boundary_gap) & (x <= front_wall_x),
        z,
        np.nan
    )

def add_dimension(ax, x1, y1, x2, y2, text, text_offset=(0, 0)):
    ax.annotate(
        "",
        xy=(x1, y1),
        xytext=(x2, y2),
        arrowprops=dict(arrowstyle="<->", linewidth=1.3)
    )

    ax.text(
        (x1 + x2) / 2 + text_offset[0],
        (y1 + y2) / 2 + text_offset[1],
        text,
        ha="center",
        va="center",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.9)
    )

# --------------------------------------------------
# Find longest board that fits below roof
# --------------------------------------------------

candidate_lengths = np.linspace(1.0, target_board_length, 1000)
best_length = None

for length in candidate_lengths:
    bx, bz, tx, tz = board_coordinates(boundary_gap, length)

    trial_front_wall_x = tx + top_clearance_to_front_wall

    board_xs = np.linspace(bx, tx, 200)
    board_zs = np.linspace(bz, tz, 200)

    roof_over_board = roof_height(board_xs, trial_front_wall_x)

    if np.any(np.isnan(roof_over_board)):
        continue

    if np.all(board_zs <= roof_over_board):
        best_length = length

if best_length is None:
    raise ValueError("No board fits below the roof with the current settings.")

# Final geometry
bx, bz, tx, tz = board_coordinates(boundary_gap, best_length)

front_wall_x = tx + top_clearance_to_front_wall
shed_depth = front_wall_x - boundary_gap

shed_x = np.linspace(boundary_gap, front_wall_x, 300)
roof_z = roof_height(shed_x, front_wall_x)
limit_z = height_limit(shed_x)

roof_exceeds_limit = np.any(roof_z > limit_z)

# Board-roof check
board_xs = np.linspace(bx, tx, 200)
board_zs = np.linspace(bz, tz, 200)
roof_over_board = roof_height(board_xs, front_wall_x)
board_exceeds_roof = np.any(board_zs > roof_over_board)

# --------------------------------------------------
# Print results
# --------------------------------------------------

print(f"Requested board length: {target_board_length:.2f} m")
print(f"Actual board length fitted: {best_length:.2f} m")
print(f"Board angle: {board_angle_deg:.1f}° overhanging")
print(f"Roof pitch: {roof_pitch_deg:.1f}°")
print(f"Shed depth required: {shed_depth:.2f} m")
print(f"Boundary gap: {boundary_gap:.2f} m")
print(f"Top-to-front-wall clearance: {top_clearance_to_front_wall:.2f} m")
print(f"Board top x-position: {tx:.2f} m from boundary")
print(f"Board top height: {tz:.2f} m")
print(f"Back wall height: {roof_height(boundary_gap, front_wall_x):.2f} m")
print(f"Front wall height: {roof_height(front_wall_x, front_wall_x):.2f} m")

if roof_exceeds_limit:
    print("WARNING: roof exceeds the planning height envelope.")
else:
    print("Roof is within the planning height envelope.")

if board_exceeds_roof:
    print("WARNING: board exceeds roof line.")
else:
    print("Board is below the roof.")

# --------------------------------------------------
# Plot
# --------------------------------------------------

x = np.linspace(0, front_wall_x + 0.75, 600)

fig, ax = plt.subplots(figsize=(14, 8))

# Planning envelope
ax.fill_between(
    x,
    0,
    height_limit(x),
    alpha=0.12,
    label="Permitted planning envelope"
)

ax.plot(
    x,
    height_limit(x),
    linestyle="--",
    linewidth=2,
    label="Planning height limit"
)

# Shed
ax.plot([boundary_gap, front_wall_x], [0, 0], linewidth=3, label="Shed floor")
ax.plot([boundary_gap, boundary_gap], [0, roof_height(boundary_gap, front_wall_x)], linewidth=3, label="Back wall")
ax.plot([front_wall_x, front_wall_x], [0, roof_height(front_wall_x, front_wall_x)], linewidth=3, label="Front wall")
ax.plot(shed_x, roof_z, linewidth=4, label=f"Roof ({roof_pitch_deg}°)")

# Boundary lines
ax.axvline(0, linestyle=":", linewidth=3, label="Boundary")
ax.axvline(boundary_limit_distance, linestyle="-.", linewidth=3, label="2 m planning line")

# Board
ax.plot([bx, tx], [bz, tz], linewidth=6, label=f"Board ({board_angle_deg}°)")
ax.plot(bx, bz, marker="o", markersize=9)
ax.plot(tx, tz, marker="s", markersize=9)

# Clearance
ax.plot(
    [tx, front_wall_x],
    [tz, tz],
    linestyle=":",
    linewidth=2,
    label="Top clearance"
)

# Dimensions
add_dimension(ax, 0, -0.25, boundary_gap, -0.25, f"boundary gap\n{boundary_gap:.2f} m")
add_dimension(ax, boundary_gap, -0.55, front_wall_x, -0.55, f"shed depth\n{shed_depth:.2f} m")
add_dimension(ax, 0, -0.85, boundary_limit_distance, -0.85, "2 m planning zone")

add_dimension(
    ax,
    bx, bz,
    tx, tz,
    f"board length\n{best_length:.2f} m",
    text_offset=(0.25, 0.25)
)

add_dimension(
    ax,
    tx, tz + 0.25,
    front_wall_x, tz + 0.25,
    f"top clearance\n{top_clearance_to_front_wall:.2f} m"
)

add_dimension(
    ax,
    boundary_gap - 0.15, 0,
    boundary_gap - 0.15, roof_height(boundary_gap, front_wall_x),
    f"back height\n{roof_height(boundary_gap, front_wall_x):.2f} m"
)

add_dimension(
    ax,
    front_wall_x + 0.18, 0,
    front_wall_x + 0.18, roof_height(front_wall_x, front_wall_x),
    f"front height\n{roof_height(front_wall_x, front_wall_x):.2f} m"
)

# Board angle arc
arc = Arc(
    (bx, bz),
    0.75,
    0.75,
    theta1=90 - board_angle_deg,
    theta2=90,
    linewidth=1.5
)
ax.add_patch(arc)

ax.text(
    bx + 0.45,
    bz + 0.50,
    f"{board_angle_deg}°\nfrom vertical",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.9)
)

# Roof pitch label
mid_x = boundary_gap + shed_depth * 0.6

ax.text(
    mid_x,
    roof_height(mid_x, front_wall_x) + 0.18,
    f"roof pitch {roof_pitch_deg}°",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.9)
)

# Warnings on plot
if roof_exceeds_limit:
    ax.text(
        0.03, 0.96,
        "WARNING: roof exceeds planning envelope",
        transform=ax.transAxes,
        fontsize=11,
        va="top",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", alpha=0.95)
    )

if board_exceeds_roof:
    ax.text(
        0.03, 0.90,
        "WARNING: board exceeds roof line",
        transform=ax.transAxes,
        fontsize=11,
        va="top",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", alpha=0.95)
    )

# Formatting
ax.set_xlabel("Distance from boundary (m)")
ax.set_ylabel("Height (m)")
ax.set_title("Climbing Shed Side Profile")

ax.set_xlim(-0.25, front_wall_x + 0.75)
ax.set_ylim(-1.05, max(far_boundary_max_height, roof_height(front_wall_x, front_wall_x)) + 0.55)

ax.set_aspect("equal", adjustable="box")
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()